# バイオ技術 1-5：機能解析入門
## DEGリストから生物学的意味を読み解く
### GO enrichment analysis & KEGG pathway analysis

これまでにPCA、DEG解析、Volcano plot、Heatmap、Candidate gene selectionまで行いました。

このNotebookではRとBioconductorを使って、

1. GO enrichment analysis
2. KEGG pathway enrichment analysis
3. 結果の可視化
4. 生物学的解釈

を行います。

## 今日の流れ
DEG list → Gene ID conversion → GO enrichment → Visualization → KEGG enrichment → Biological interpretation

### このNotebookはRで実行します
Google ColabでR kernelとして開いて実行してください。


---
# 0. 研究背景の確認

今回のデータは、ヒト気道平滑筋細胞のUntreatedとDexamethasone（Dex）を比較したRNA-seqデータです。

今回の問いは、

> **Dexによって上昇した遺伝子群には、どのような生物学的機能が共通しているか？**

です。

## 予想してみよう
Dexはglucocorticoidです。どのような生物学的機能やpathwayが変化していそうでしょうか？

1.
2.
3.


---
# 1. 必要なpackageをinstallする

初回のみinstallに時間がかかります。

使用する主なpackage：
- DESeq2
- clusterProfiler
- enrichplot
- org.Hs.eg.db


In [ ]:
if (!requireNamespace("BiocManager", quietly = TRUE)) {
    install.packages("BiocManager")
}

packages <- c(
    "DESeq2",
    "clusterProfiler",
    "enrichplot",
    "org.Hs.eg.db"
)

for (pkg in packages) {
    if (!requireNamespace(pkg, quietly = TRUE)) {
        BiocManager::install(pkg, ask = FALSE, update = FALSE)
    }
}

if (!requireNamespace("ggplot2", quietly = TRUE)) {
    install.packages("ggplot2")
}

cat("Package installation completed.\n")


---
# 2. packageを読み込む


In [ ]:
library(DESeq2)
library(clusterProfiler)
library(enrichplot)
library(org.Hs.eg.db)
library(ggplot2)

cat("Packages loaded.\n")


---
# 3. 解析データを読み込む

前の演習と同じairway RNA-seq datasetを使います。
このNotebook単独でも実行できるよう、DESeq2 objectをdownloadします。


In [ ]:
DDS_URL <- paste0(
    "https://raw.githubusercontent.com/",
    "mandhri/airway-glucocorticoid-rnaseq/",
    "main/results/deseq2/dds_raw.rds"
)

download.file(
    DDS_URL,
    destfile = "dds_raw.rds",
    mode = "wb"
)

dds <- readRDS("dds_raw.rds")

dds


### designを確認する

今回は `~ cell + dex` というdesignです。

これは、

> donorの違いを考慮しながらDex処理効果を見る

という考え方です。


In [ ]:
design(dds)


---
# 4. DESeq2を実行する

機能解析に使うDEG listを作るためにDESeq2を実行します。

※ このNotebookの中心はDEG解析そのものではなく、その後の機能解析です。


In [ ]:
dds <- DESeq(dds)

res <- results(
    dds,
    contrast = c("dex", "trt", "untrt"),
    alpha = 0.05
)

res_df <- as.data.frame(res)
res_df$ensembl_id <- rownames(res_df)

head(res_df)


---
# 5. Dexで上昇したDEGを抽出する

前の演習と同じ考え方で、

- log2FC > 2.0
- adjusted p-value < 0.05

を満たすDex-induced genesを抽出します。


In [ ]:
LOG2FC_THRESHOLD <- 2.0
PADJ_THRESHOLD <- 0.05

dex_deg <- subset(
    res_df,
    !is.na(padj) &
    log2FoldChange > LOG2FC_THRESHOLD &
    padj < PADJ_THRESHOLD
)

dex_deg <- dex_deg[order(dex_deg$padj), ]

cat("Number of Dex-induced DEGs:", nrow(dex_deg), "\n")

head(dex_deg, 10)


### ミニ演習1

1. Dex-induced DEGは何個ありましたか？
2. log2FC thresholdを1.0にすると、DEG数はどうなると思いますか？
3. 閾値を厳しくすると、機能解析結果にはどのような影響があると思いますか？

答え：

1.
2.
3.


---
# 6. Ensembl IDをEntrez IDへ変換する

GO解析やKEGG解析では、gene ID形式を揃える必要があります。

今回は、

```text
ENSEMBL
↓
ENTREZID / SYMBOL
```

へ変換します。

### 重要：Ensembl versionを削除する

airway datasetのEnsembl IDには、

```text
ENSG00000103196.14
```

のようにversion番号が付く場合があります。

一方、`org.Hs.eg.db`では、

```text
ENSG00000103196
```

のようなversionなしIDを使います。

そのため、ID変換の前に末尾の`.14`などを削除します。


In [ ]:
# airway datasetのEnsembl IDには、
# ENSG00000123456.12 のようなversion suffixが付いています。
# org.Hs.eg.dbで対応できるよう、末尾の ".数字" を削除します。

dex_deg$ensembl_clean <- sub(
    "\\..*$",
    "",
    dex_deg$ensembl_id
)

cat("Original IDs:\n")
print(
    head(
        dex_deg$ensembl_id
    )
)

cat("\nVersion-stripped IDs:\n")
print(
    head(
        dex_deg$ensembl_clean
    )
)

gene_map <- bitr(
    unique(
        dex_deg$ensembl_clean
    ),
    fromType = "ENSEMBL",
    toType = c(
        "ENTREZID",
        "SYMBOL"
    ),
    OrgDb = org.Hs.eg.db
)

cat(
    "\nInput genes:",
    length(
        unique(
            dex_deg$ensembl_clean
        )
    ),
    "\n"
)

cat(
    "Mapped genes:",
    nrow(gene_map),
    "\n"
)

head(gene_map)


### 考えてみよう

すべてのgeneが必ず1対1で変換されるとは限りません。

理由として、
- annotation versionの違い
- 1対多対応
- database間の対応関係

などがあります。

機能解析では、**ID conversionも重要な解析工程**です。


In [ ]:
entrez_genes <- unique(gene_map$ENTREZID)
length(entrez_genes)


---
# 7. GO enrichment analysis

Gene Ontology（GO）は、gene productの機能を体系的に整理したものです。

主に、
- BP：Biological Process
- MF：Molecular Function
- CC：Cellular Component

があります。

今回はまずBiological Process（BP）を解析します。


In [ ]:
ego_bp <- enrichGO(
    gene = entrez_genes,
    OrgDb = org.Hs.eg.db,
    keyType = "ENTREZID",
    ont = "BP",
    pAdjustMethod = "BH",
    pvalueCutoff = 0.05,
    qvalueCutoff = 0.20,
    readable = TRUE
)

ego_bp


---
# 8. GO結果を表で見る


In [ ]:
go_table <- as.data.frame(ego_bp)

head(go_table, 15)


## GO結果の主な列

- `ID`：GO ID
- `Description`：GO term名
- `GeneRatio`：入力geneのうちtermに含まれる割合
- `BgRatio`：background geneにおける割合
- `p.adjust`：多重検定補正後p値
- `Count`：termに含まれたgene数

### ミニ演習2
1. 一番上のGO termは何ですか？
2. そのtermは今回のDex処理と関係しそうですか？
3. 似た意味のGO termが複数ありますか？

答え：

1.
2.
3.


---
# 9. GO enrichmentをdotplotで見る

dotplotでは一般に、
- y軸：GO term
- x軸：GeneRatio
- point size：gene count
- color：adjusted p-value

を表します。


In [ ]:
dotplot(
    ego_bp,
    showCategory = 15
) +
    ggtitle("GO Biological Process Enrichment")


### ミニ演習3：dotplotを読む

1. GeneRatioが大きいtermはどれですか？
2. Countが多いtermはどれですか？
3. adjusted p-valueが小さいtermはどれですか？
4. 「最も上にあるterm = 最も重要」と言い切ってよいでしょうか？

答え：

1.
2.
3.
4.


---
# 10. GO enrichmentをbarplotで見る


In [ ]:
barplot(
    ego_bp,
    showCategory = 15
) +
    ggtitle("Top GO Biological Processes")


---
# 11. GO termの重複を考える

GOでは意味の近いtermが多数並ぶことがあります。

これはGOが階層構造を持っているためです。

したがって、

> 有意なtermを1つずつ独立に読む

だけではなく、

> **共通する生物学的テーマをまとめて考える**

ことが重要です。

### 自分でまとめてみよう

Theme 1：

Theme 2：

Theme 3：


---
# 12. KEGG pathway enrichment analysis

次にKEGG pathway enrichmentを行います。

GOがgene functionを体系的に整理するのに対して、
KEGGでは、

> **geneがどのpathwayに集まっているか**

を見ます。


In [ ]:
ekegg <- tryCatch(
    {
        enrichKEGG(
            gene = entrez_genes,
            organism = "hsa",
            pvalueCutoff = 0.05,
            pAdjustMethod = "BH"
        )
    },
    error = function(e) {
        message("KEGG analysis failed: ", e$message)
        NULL
    }
)

ekegg


### 注意

KEGG解析では外部databaseへaccessするため、
一時的なnetwork errorが起こる場合があります。

その場合は少し時間を置いて再実行してください。


---
# 13. KEGG結果を表で見る


In [ ]:
if (!is.null(ekegg)) {
    kegg_table <- as.data.frame(ekegg)
    print(head(kegg_table, 15))
}


### ミニ演習4

1. 上位のKEGG pathwayは何ですか？
2. そのpathwayはDex作用と関係しそうですか？
3. GO解析結果と共通する生物学的テーマはありますか？

答え：

1.
2.
3.


---
# 14. KEGG結果をdotplotで見る


In [ ]:
if (!is.null(ekegg) && nrow(as.data.frame(ekegg)) > 0) {
    dotplot(
        ekegg,
        showCategory = 15
    ) +
        ggtitle("KEGG Pathway Enrichment")
}


---
# 15. GOとKEGGを比較する

## GO Biological Process
どのような生物学的processに関係するgeneが多いか？

## KEGG pathway
どのような既知のbiological pathwayにgeneが集まっているか？

| 観点 | GO BP | KEGG |
|---|---|---|
| 上位結果 | | |
| 共通テーマ | | |
| Dexとの関係 | | |
| 解釈しやすさ | | |


---
# 16. 候補geneと機能解析をつなげる

1-4では個々のcandidate geneに注目しました。
1-5ではgene set全体を見ています。

Individual candidate gene
+
Enriched biological function
+
Known pathway
↓
Biological hypothesis

### 考えてみよう

CRISPLD2のような個別gene候補と、
今回のGO / KEGG enrichment結果を、
どのようにつなげて説明できそうですか？

答え：


---
# 17. 次に行うwet実験を考える

例えば、
- qPCRで複数geneを検証する
- protein levelを確認する
- cytokine productionを測定する
- gene knockdownを行う
- pathway inhibitorを使う
- 別donorで再現性を見る

などが考えられます。

## あなたの実験計画

### 注目するGO term / pathway
名称：

### 仮説

### 次に行うwet実験
1.
2.
3.

### その実験で何がわかるか


---
# 18. まとめ

今日の演習では、

1. Dex-induced DEGを抽出した
2. Ensembl IDをEntrez IDへ変換した
3. GO Biological Process enrichmentを行った
4. dotplot / barplotで結果を可視化した
5. KEGG pathway enrichmentを行った
6. GOとKEGGを比較した
7. 生物学的仮説と次のwet実験を考えた

## 今日一番伝えたいこと

DEG解析では、

> **どのgeneが変化したか**

を見ます。

機能解析では、

> **変化したgene群に、どのような生物学的意味があるか**

を考えます。

RNA-seq → DEG analysis → GO / Pathway analysis → Biological hypothesis → Next wet experiment


---
# 19. 発展課題

### 1. Up-regulated geneとDown-regulated geneを分ける
異なる機能がenrichされる可能性があります。

### 2. GO BP以外を調べる
- MF：Molecular Function
- CC：Cellular Component

### 3. GSEAを行う
DEG cutoffを超えたgeneだけでなく、**全geneのranking**を利用します。

### 最後の問い

ORAとGSEAではどちらが良いのでしょうか？

答えは、

> **研究目的とデータによって使い分ける**

です。
